# The Infosiast — Local LLM VRAM, KV-Cache & Quantization Benchmark
**Prepared by [Kshitij Gupta](https://www.linkedin.com/in/kshitij-gupta-855535134/) for [The Infosiast](https://theinfosiast.com/)**

This reproducible Google Colab notebook accompanies:
* [Running Local LLMs on Consumer Hardware in 2026](https://theinfosiast.com/running-local-llms-consumer-hardware-2026/)
* [DeepSeek R1 Architecture & Local Deployment Guide](https://theinfosiast.com/deepseek-r1-architecture-deployment-guide/)

Run all cells on a Google Colab **T4 / L4 GPU** (using your 200 monthly Google AI Pro CCUs) to calculate exact VRAM footprints across `FP16`, `Q8_0`, `Q5_K_M`, and `Q4_K_M`, measure MLA vs standard GQA KV-cache scaling, and benchmark live matrix-multiplication memory bandwidth.


### Step 1: Deterministic VRAM Footprint Calculator (Weights + KV Cache + Runtime Buffer)


In [ ]:
import pandas as pd

def estimate_vram_gb(params_b: float, bits_per_weight: float, ctx_tokens: int, kv_bits: int = 8, mla_compression: float = 1.0, buffer_gb: float = 0.65) -> dict:
    weight_gb = (params_b * 1e9 * (bits_per_weight / 8.0)) / (1024**3)
    kv_gb = ((ctx_tokens * 0.00012 * (params_b / 14.0)) * (kv_bits / 16.0)) * mla_compression
    total_gb = round(weight_gb + kv_gb + buffer_gb, 2)
    return {
        'Model Size (B)': params_b,
        'QuantizationBits': bits_per_weight,
        'Context Window': ctx_tokens,
        'Weights (GB)': round(weight_gb, 2),
        'KV Cache (GB)': round(kv_gb, 2),
        'Total VRAM (GB)': total_gb,
        'Fits 12GB GPU': 'YES' if total_gb <= 11.5 else 'NO',
        'Fits 16GB GPU': 'YES' if total_gb <= 15.5 else 'NO',
        'Fits 24GB GPU': 'YES' if total_gb <= 23.5 else 'NO',
    }

scenarios = [
    estimate_vram_gb(8.0, 4.5, 8192, kv_bits=8),
    estimate_vram_gb(14.0, 4.5, 8192, kv_bits=8),
    estimate_vram_gb(14.0, 5.5, 16384, kv_bits=8),
    estimate_vram_gb(32.0, 4.5, 16384, kv_bits=8),
    estimate_vram_gb(32.0, 4.5, 32768, kv_bits=8, mla_compression=0.25),
    estimate_vram_gb(70.0, 4.5, 16384, kv_bits=8),
]
df = pd.DataFrame(scenarios)
print(df.to_string(index=False))


### Step 2: Live GPU / CPU Memory-Bandwidth & Token Throughput Probe


In [ ]:
import time
import torch

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Active Compute Device: {device.upper()}')
if device == 'cuda':
    print('GPU Device Name:', torch.cuda.get_device_name(0))

dim = 4096
dtype = torch.float16 if device == 'cuda' else torch.float32
w = torch.randn((dim, dim), device=device, dtype=dtype)
x = torch.randn((1, dim), device=device, dtype=dtype)

if device == 'cuda':
    torch.cuda.synchronize()
t0 = time.perf_counter()
iters = 500
for _ in range(iters):
    y = torch.matmul(x, w)
if device == 'cuda':
    torch.cuda.synchronize()
elapsed = time.perf_counter() - t0
bytes_moved = iters * dim * dim * (2 if dtype == torch.float16 else 4)
gbps = (bytes_moved / elapsed) / (1024**3)
print(f'Measured Effective Matrix Read Bandwidth: {gbps:.2f} GB/s ({elapsed*1000:.2f} ms for {iters} passes)')
